In [2]:
from collections import OrderedDict

import seaborn as sns
import matplotlib.pylab as plt

import ipywidgets as widgets

import plotly.graph_objects as go

import pandas as pd
import numpy as np

from bqplot import *
from datetime import datetime, date
from pandas.tseries.offsets import BDay

import warnings
warnings.filterwarnings('ignore')

from ipydatagrid import DataGrid, TextRenderer


In [3]:
bdTickers = ['DBR','FRTR','BTPS','EIB','EU','ESM','CADES','UNEDIC','KFW','RENTEN','NRW']
bdBuckets=['1Y','2Y','3Y','4Y','5Y','6Y','7Y','8Y','9Y','10Y','12Y-']

def get_date_range():
    today = pd.Timestamp.today()
    start_date = (today - pd.offsets.BMonthBegin(60))
    end_date = (today - pd.offsets.BDay(2))
    date_range = pd.date_range(start=start_date, end=end_date, freq='D')
    return [date.strftime('%Y-%m-%d') for date in date_range]

date_range = get_date_range()

def getBondsUniverse(aCurrency):
    issuers =[]
    if aCurrency == 'USD':
        issuers.append('7915Z US Equity') #IBRD
        issuers.append('1501510D US Equity') #UST
        issuers.append('1501512D US Equity') #T
        issuers.append('654429Z LX Equity') #EIB
        issuers.append('345300Z BB Equity') #EU
        issuers.append('211430Z FP Equity') #CADES
        issuers.append('223338Z FP Equity') #AGFRNC
        issuers.append('0570154D LX Equity') #ESM
        # issuers.append('KFW AU Equity') #KFW
        issuers.append('3413Z GR Equity') #All german bonds
        issuers.append('220055Z FP Equity') #UNEDIC
        issuers.append('2518Z GR Equity') #RENTEN
        issuers.append('46716Z GR Equity') #NRW
        issuers.append('NRWB GR Equity') #NRWBK

        issuers.append('4388Z NA Equity') #BNG
        issuers.append('8163317Z BB Equity') #FLEMSH
        issuers.append('1062Z NO Equity') #KBN
        issuers.append('140362Z SS Equity') #KOMINS
        issuers.append('69761Z DC Equity') #KOMMUN

        issuers.append('BNS CN Equity') #BNS
        issuers.append('BMO CN Equity') #BMO
        issuers.append('CM CN Equity') #CIBC
        
    elif aCurrency == 'GBP':
        issuers.append('6152Z LN Equity') #IGILTS
     
        issuers.append('211430Z FP Equity') #CADES
        issuers.append('3413Z GR Equity') #All german bonds
        issuers.append('2518Z GR Equity') #RENTEN
        issuers.append('46716Z GR Equity') #NRW
        issuers.append('NRWB GR Equity') #NRWBK
        issuers.append('81074Z PM Equity') #ASIA
        issuers.append('7915Z US Equity') #IBRD
        issuers.append('654429Z LX Equity') #EIB
    
    elif aCurrency == 'JPY':
        issuers.append('1319293D JP Equity') #JGB 
        issuers.append('JAFZ JP Equity') #JFM
        issuers.append('8684034Z JP Equity') #JICA
        
    elif aCurrency == 'CZK':  
        issuers.append('1040Z CP Equity') #CZGB
        issuers.append('7915Z US Equity') #IBRD

    elif aCurrency == 'EUR':
        issuers.append('345300Z BB Equity') #EU
        issuers.append('211430Z FP Equity') #CADES
        issuers.append('223338Z FP Equity') #AGFRNC
        issuers.append('0570154D LX Equity') #ESM
        # issuers.append('KFW AU Equity') #KFW
        issuers.append('3413Z GR Equity') #All german bonds
        issuers.append('220055Z FP Equity') #UNEDIC
        issuers.append('2518Z GR Equity') #RENTEN
        issuers.append('46716Z GR Equity') #NRW
        issuers.append('NRWB GR Equity') #NRWBK

        issuers.append('4388Z NA Equity') #BNG
        issuers.append('8163317Z BB Equity') #FLEMSH
        issuers.append('1062Z NO Equity') #KBN
        issuers.append('140362Z SS Equity') #KOMINS
        issuers.append('69761Z DC Equity') #KOMMUN

        issuers.append('BNS CN Equity') #BNS
        issuers.append('BMO CN Equity') #BMO
        issuers.append('CM CN Equity') #CIBC
        issuers.append('1501888D FP Equity') #FRTR
        issuers.append('0938298D GR Equity') #DBR
        issuers.append('2103Z IM Equity') #BTPS
        issuers.append('111136Z BB Equity') #BGB
        issuers.append('1841Z SM Equity') #SPGB
        issuers.append('1533Z NA Equity') #NETHER
        issuers.append('4105317Z LX Equity') #EFSF
        issuers.append('223727Z FP Equity') #BTF
        issuers.append('654429Z LX Equity') #EIB
    univ = bq.univ.bonds(issuers)

    # Defining filters
    filter_Currency = bq.data.crncy() == aCurrency
    filter_Maturity_Low= bq.data.maturity() > '360d'
    filter_Maturity_High=bq.data.maturity() < '12Y'
    filter_CouponType = bq.data.cpn_typ() == 'FIXED'
    filter_InflationFlag = bq.data.inflation_linked_indicator() == False
    filter_IssuanceSize = bq.data.amt_issued() >= 1*10**9
    
    filter_PaymentRank = bq.data.payment_rank() == 'Secured'
    filter_BicsCorporate = bq.data.BICS_LEVEL_1_SECTOR_CODE() == 35
    filter_BicsNotCorporate = bq.data.BICS_LEVEL_1_SECTOR_CODE() != 35
    
    
    # Creating criterias object
    filters = bq.func.and_(filter_Currency,filter_Maturity_Low)
    filters = bq.func.and_(filters,filter_Maturity_High)
    filters = bq.func.and_(filters,filter_CouponType)
    filters = bq.func.and_(filters,filter_InflationFlag)
    filters = bq.func.and_(filters,filter_IssuanceSize)
    
    filtersCovered = bq.func.and_(filter_PaymentRank,filter_BicsCorporate)
    filtersCovered = bq.func.or_(filtersCovered,filter_BicsNotCorporate)
    
    filters = bq.func.and_(filters,filtersCovered)

    
    # filters = bq.func.and_(filters,filter_PaymentRank)
    # filters = bq.func.and_(filters,filter_BicsCorporate)
    
    # Filtering the universe
    univ_filtered = bq.univ.filter(univ,filters)
    
    return univ_filtered

In [4]:
def getBondsUniverseTIPS(aCurrency):

    # Defining Universe
    issuers = [] #EIB

    
    if aCurrency == 'USD':
        issuers.append('1501512D US Equity') #T 
    elif aCurrency == 'GBP':
        issuers.append('1501825D LN Equity')

    univ = bq.univ.bonds(issuers)

    # Defining filters
    filter_Currency = bq.data.crncy() == aCurrency
    filter_Maturity_Low= bq.data.maturity() > '360d'
    filter_Maturity_High=bq.data.maturity() < '12Y'
    filter_CouponType = bq.data.cpn_typ() == 'FIXED'
    filter_InflationFlag = bq.data.inflation_linked_indicator() == True 
    filter_IssuanceSize = bq.data.amt_issued() >= 1*10**9
    
    filter_PaymentRank = bq.data.payment_rank() == 'Secured'
    filter_BicsCorporate = bq.data.BICS_LEVEL_1_SECTOR_CODE() == 35
    filter_BicsNotCorporate = bq.data.BICS_LEVEL_1_SECTOR_CODE() != 35
    
    
    # Creating criterias object
    filters = bq.func.and_(filter_Currency,filter_Maturity_Low)
    filters = bq.func.and_(filters,filter_Maturity_High)
    filters = bq.func.and_(filters,filter_CouponType)

    filters = bq.func.and_(filters,filter_IssuanceSize)
    
    filtersCovered = bq.func.and_(filter_PaymentRank,filter_BicsCorporate)
    filtersCovered = bq.func.or_(filtersCovered,filter_BicsNotCorporate)
    
    filters = bq.func.and_(filters,filtersCovered)

    
    # filters = bq.func.and_(filters,filter_PaymentRank)
    # filters = bq.func.and_(filters,filter_BicsCorporate)
    
    # Filtering the universe
    univ_filtered = bq.univ.filter(univ,filters)
    
    return univ_filtered

In [5]:
def getBondsFields(aPastDate,aCurrency,aXccy, aSource='BGN'):

    # Defining fields to retrieve and inserting into dictionnary
    flds = OrderedDict()

    bdISIN = bq.data.id_isin()
    flds['ISIN'] = bdISIN

    bdTicker = bq.data.ticker()
    flds['Ticker'] = bdTicker

    bdSecurityDes = bq.data.security_des()
    flds['Description'] = bdSecurityDes
    
    bdIssueDate = bq.data.issue_dt()
    flds['Issue_Date'] = bdIssueDate
    
    bdAmtOutstanding = bq.data.amt_outstanding() / 10**6
    flds['Amt_outstanding'] = bdAmtOutstanding
    
    bdMaturity = bq.data.maturity()
    flds['Maturity'] = bdMaturity

    bqprice = bq.data.price()
    flds['Price'] = bqprice
    
    bdYield_Current = bq.data.yield_(pricing_source=aSource,side='mid')
    flds['Yield_C'] = bdYield_Current
    
    bdYield_Past = bq.data.yield_(dates=aPastDate,pricing_source=aSource,side='mid')
    flds['Yield_P'] = bdYield_Past
    
    TII_Current = bq.data.spread(spread_type='ASW') # For TIPS 
    flds['ASW_TII'] = TII_Current

    TII_Past = bq.data.spread(spread_type='ASW', dates=aPastDate) # For TIPS
    flds['ASW_TII_PAST'] = TII_Past

    if aCurrency == 'EUR':
        bdGspread_Current = bq.data.spread(spread_type='G',CURVE_ID='I16')  
        flds['G_Spread'] =  bdGspread_Current
        
        bdGspread_Current_I13 = bq.data.spread(spread_type='G',side = 'mid', pricing_source='BGN')  
        flds['G_Spread_I13'] =  bdGspread_Current_I13
        
        bdGspread_Past = bq.data.spread(spread_type='G', dates=aPastDate ,side = 'mid', pricing_source='BGN') 
        flds['G_Spread_Past'] =  bdGspread_Past
        
    else :
        bdGspread_Current = bq.data.spread(spread_type='G')  
        flds['G_Spread'] =  bdGspread_Current

        bdGspread_Past = bq.data.spread(spread_type='G', dates=aPastDate) 
        flds['G_Spread_Past'] =  bdGspread_Past

    if aXccy is not None :
        aXccy = aXccy[:3] 
        bd_Xccy = bq.data.yield_(cross_currency = aXccy) 
        flds['Xccy'] =  bd_Xccy
        
        bd_Xccy_Past = bq.data.yield_(cross_currency = aXccy) 
        flds['Xccy_Past'] =  bd_Xccy_Past
        if aCurrency == 'EUR' :
            bd_i_spread = bq.data.spread(spread_type='i', CURVE_ID='S514') 
            flds['i_spread'] =  bd_i_spread   
        else :
            bd_i_spread = bq.data.spread(spread_type='i') 
            flds['i_spread'] =  bd_i_spread 
    
    
    bdDate_Current = bq.data.yield_(pricing_source=aSource)['DATE']
    flds['Date_C'] = bdDate_Current
    
    bdDate_Past = bq.data.yield_(dates=aPastDate,pricing_source=aSource)['DATE']
    flds['Date_P'] = bdDate_Past
    
    diffYield = bdYield_Current - bdYield_Past
    flds['Yield_Delta'] = (diffYield*100).round(2)

    bdFlag_Green = bq.data.green_bond_loan_indicator()
    flds['Green'] = bdFlag_Green

    bdFlag_Social = bq.data.social_bond_ind()
    flds['Social'] = bdFlag_Social

    bdFlag_Sustainable = bq.data.sustainability_bond_ind()
    flds['Sustainable'] = bdFlag_Sustainable
    
    bdAxes_Bid = bq.data.axes()['bid_total_size'] / 10**6
    flds['Axes_Bid'] = bdAxes_Bid
    
    bdAxes_Ask = bq.data.axes()['ask_total_size'] / 10**6
    flds['Axes_Ask'] = bdAxes_Ask

    bdResidualMaturity = bdMaturity-bq.func.today()
    flds['ResidualMaturity'] = bdResidualMaturity
    
    bdResidualMaturity_Past = bdMaturity-bdDate_Past
    flds['ResidualMaturity_Past'] = bdResidualMaturity_Past

    bdTenor=(bdResidualMaturity/365.25).round(0)
    bdBins=bq.func.bins(bdTenor, bins=[1.5,2.5,3.5,4.5,5.5,6.5,7.5,8.5,9.5,10.5], bin_names=bdBuckets)
    flds['Bucket'] = bdBins
    
    bdTenor_Past=(bdResidualMaturity_Past/365.25).round(0)
    bdBins=bq.func.bins(bdTenor_Past, bins=[1.5,2.5,3.5,4.5,5.5,6.5,7.5,8.5,9.5,10.5], bin_names=bdBuckets)
    flds['Bucket_Past'] = bdBins
    
    return flds


In [6]:
def runRequest(aUniverse, aFields):
    # Preparing request
    request =  bql.Request(aUniverse, aFields, with_params={'aggregateby': 'security'})

    # Sending request
    response = bq.execute(request)

    # Inserting results into DF
    tbl = pd.DataFrame({r.name:r.df()[r.name] for r in response})
    return tbl

def getBonds(aPastDate,aCurrency,aXccy):
    if aCurrency != 'USD' and aCurrency != 'GBP' :
        univ = getBondsUniverse(aCurrency)
        flds = getBondsFields(aPastDate,aCurrency,aXccy)
        return runRequest(univ,flds)
    univ = getBondsUniverse(aCurrency)
    univ1 = getBondsUniverseTIPS(aCurrency)
    flds = getBondsFields(aPastDate,aCurrency,aXccy)
    df = runRequest(univ1,flds)
    dt = runRequest(univ,flds)
    return pd.concat([df, dt], ignore_index=True, sort=False)

In [7]:
def getSwaps(aIndex, aPastDate):

    universe = bq.univ.members(aIndex +' Index',type='curve_tenors')

    flds = OrderedDict()
    flds['tenor'] = bq.data.id()['tenor']
    flds['rate'] = bq.data.curve_rate(side='mid')
    flds['rate_Past'] = bq.data.curve_rate(side='mid',dates=aPastDate)
    req = bql.Request(universe, flds)
    res = bq.execute(req)
    
    df = pd.DataFrame({r.name:r.df()[r.name] for r in res})

    dfDay = df[df['tenor'].str.contains('D')]
    dfDay['Duration'] = (dfDay['tenor'].str[:-1]).astype(int)/365

    dfWeek = df[df['tenor'].str.contains('W')]
    dfWeek['Duration'] = (dfWeek['tenor'].str[:-1]).astype(int)/52

    dfMonth = df[df['tenor'].str.contains('M')]
    dfMonth['Duration'] = (dfMonth['tenor'].str[:-1]).astype(int)/12

    dfYear = df[df['tenor'].str.contains('Y')]
    dfYear['Duration'] = (dfYear['tenor'].str[:-1]).astype(int)

    df = pd.concat([dfDay,dfWeek,dfMonth,dfYear])
    df['Duration'] = df['Duration']*365
    
    df = df.sort_values(by='Duration')
    
    return df

In [8]:
def getSwaps(aIndex, aPastDate):

    universe = bq.univ.members(aIndex +' Index',type='curve_tenors')

    flds = OrderedDict()
    flds['tenor'] = bq.data.id()['tenor']
    flds['rate'] = bq.data.curve_rate(side='mid')
    flds['rate_Past'] = bq.data.curve_rate(side='mid',dates=aPastDate)
    req = bql.Request(universe, flds)
    res = bq.execute(req)
    
    df = pd.DataFrame({r.name:r.df()[r.name] for r in res})

    dfDay = df[df['tenor'].str.contains('D')]
    dfDay['Duration'] = (dfDay['tenor'].str[:-1]).astype(int)/365

    dfWeek = df[df['tenor'].str.contains('W')]
    dfWeek['Duration'] = (dfWeek['tenor'].str[:-1]).astype(int)/52

    dfMonth = df[df['tenor'].str.contains('M')]
    dfMonth['Duration'] = (dfMonth['tenor'].str[:-1]).astype(int)/12

    dfYear = df[df['tenor'].str.contains('Y')]
    dfYear['Duration'] = (dfYear['tenor'].str[:-1]).astype(int)

    df = pd.concat([dfDay,dfWeek,dfMonth,dfYear])
    df['Duration'] = df['Duration']*365
    
    df = df.sort_values(by='Duration')
    
    return df

In [9]:
def computeISpread(aBonds,aSwaps,aSpreadType):
    if aSpreadType != 'XCCY ASW' :
        aBonds['InterpolatedSwap']= np.interp(aBonds['ResidualMaturity'],aSwaps['Duration'],aSwaps['rate'])
        aBonds['ASW'] = aBonds.apply(lambda row: row['Yield_C'] - row['InterpolatedSwap'] if str(row['Ticker']) in ['BNS','BMO','CM'] else row['ASW_TII']/100 if str(row['Ticker']) in ['TII','UKTI'] else row['Yield_C'] -row['InterpolatedSwap'] ,axis=1) 
        aBonds['ASW'] = aBonds['ASW'] * 100
        aBonds['ASW'] = aBonds['ASW'].round(2)

        aBonds['InterpolatedSwap_Past']= np.interp(aBonds['ResidualMaturity_Past'],aSwaps['Duration'],aSwaps['rate_Past'])
        aBonds['ASW_Past'] = aBonds.apply(lambda row: row['Yield_P'] - row['InterpolatedSwap_Past'] if str(row['Ticker']) in ['BNS','BMO','CM'] else row['ASW_TII_PAST']/100  if str(row['Ticker']) in ['TII'] else row['Yield_P']-row['InterpolatedSwap_Past'] ,axis=1) 
        aBonds['ASW_Past'] = aBonds['ASW_Past'] * 100
        aBonds['ASW_Past'] = aBonds['ASW_Past'].round(2)

        aBonds['ASW_Diff'] = aBonds['ASW'] - aBonds['ASW_Past']
        try : 
            aBonds['G_Spread_diff'] =  aBonds['G_Spread_I13'] - aBonds['G_Spread_Past']
        except :
            aBonds['G_Spread_diff'] =  aBonds['G_Spread'] - aBonds['G_Spread_Past']    


        aBonds = aBonds.sort_values(by='Maturity')
        
        
    else :
        aBonds = aBonds [(aBonds['Price']<=100.8) & (aBonds['Price']>=99.2)]
        aBonds['InterpolatedSwap']= np.interp(aBonds['ResidualMaturity'],aSwaps['Duration'],aSwaps['rate'])
        aBonds['Xccy'] = aBonds['Xccy'] - aBonds['InterpolatedSwap'] 
        aBonds['Xccy'] = aBonds['Xccy'] * 100
        aBonds['Xccy'] = aBonds['Xccy'].round(2)
        

        aBonds['InterpolatedSwap_Past']= np.interp(aBonds['ResidualMaturity_Past'],aSwaps['Duration'],aSwaps['rate_Past'])
        aBonds['Xccy_Past'] = aBonds['Xccy_Past'] - aBonds['InterpolatedSwap_Past'] 
        aBonds['Xccy_Past'] = aBonds['Xccy_Past'] * 100
        aBonds['Xccy_Past'] = aBonds['Xccy_Past'].round(2)

        aBonds['Xccy_Diff'] = aBonds['Xccy'] - aBonds['Xccy_Past']
        try : 
            aBonds['G_Spread_diff'] =  aBonds['G_Spread_I13'] - aBonds['G_Spread_Past']
        except :
            aBonds['G_Spread_diff'] =  aBonds['G_Spread'] - aBonds['G_Spread_Past']    


        aBonds = aBonds.sort_values(by='Maturity')
        
        
    return aBonds

In [10]:
def aggregateBonds(aBonds,aType,aCurncy,atickers,aSpreadType,aRelative,aXccyValue):
    SSA = {
        'FRTR': ['AGFRNC','CADES','UNEDIC'],
        'NETHER':['BNG'],
        'BGB':['FLEMSH']
            }
    Gov = ['DBR','NETHER','FRTR','BGB','SPGB','BTPS']
    Others = ['KOMINS','KOMMUN','BMO','BNS','CM']
    bdTickers2 = dict(zip(bdTickers, range(len(bdTickers))))

    if aSpreadType == 'ASW' :
        if aType == 'Current':
            df = aBonds.groupby(['Ticker', 'Bucket']).mean()['ASW'].round(1).unstack(fill_value=np.nan,level=1).transpose()
        else:
            df = aBonds.groupby(['Ticker', 'Bucket']).mean()['ASW_Diff'].round(1).unstack(fill_value=np.nan,level=1).transpose()
            
    if aSpreadType == 'ASW Level Relative to' :
        if aType == 'Current':
            dx = Maximum_ASW(aCurncy,aRelative)
            if aCurncy == 'EUR':
                df = dx * 100
            else :
                df = aBonds.groupby(['Ticker', 'Bucket']).mean()['ASW'].round(2).unstack(fill_value=np.nan,level=1).transpose()
                df = df / dx
                df = df * 100
        else :
            df = aBonds.groupby(['Ticker', 'Bucket']).mean()['ASW_Diff'].round(2).unstack(fill_value=np.nan,level=1).transpose()
        
    elif aSpreadType == 'G Spread' and aCurncy =='EUR' :
        if aType == 'Current':
            df = aBonds.groupby(['Ticker', 'Bucket']).mean()['G_Spread'].round(1).unstack(fill_value=np.nan,level=1).transpose()
        else:
            df = aBonds.groupby(['Ticker', 'Bucket']).mean()['G_Spread_diff'].round(1).unstack(fill_value=np.nan,level=1).transpose()
        for i in SSA :
            for j in SSA[i] :
                df[j] = (df[j] - df[i]).round(2)
        for g in Gov :
            df[g] = np.nan
        for g in Others :
            df[g] = np.nan
    
    elif aSpreadType == 'G Spread' and aCurncy =='USD':
        if aType == 'Current':
            df = aBonds.groupby(['Ticker', 'Bucket']).mean()['G_Spread'].round(1).unstack(fill_value=np.nan,level=1).transpose()
            dx = aBonds[aBonds['Ticker'].isin(['T', 'TII'])]
            dx = dx.groupby(['Ticker', 'Bucket']).mean()['ASW'].round(1).unstack(fill_value=np.nan,level=1).transpose()
            df['TII'] = (dx['TII'] - dx['T']).round(2)
        else:
            df = aBonds.groupby(['Ticker', 'Bucket']).mean()['G_Spread_diff'].round(1).unstack(fill_value=np.nan,level=1).transpose()         

        
    elif aSpreadType == 'G Spread' and aCurncy =='GBP':
        if aType == 'Current':
            df = aBonds.groupby(['Ticker', 'Bucket']).mean()['G_Spread'].round(1).unstack(fill_value=np.nan,level=1).transpose()
            dx = aBonds[aBonds['Ticker'].isin(['UKT', 'UKTI'])]
            dx = dx.groupby(['Ticker', 'Bucket']).mean()['ASW'].round(1).unstack(fill_value=np.nan,level=1).transpose()
            df['UKTI'] = (dx['UKTI'] - dx['UKT']).round(2)
        else:
            df = aBonds.groupby(['Ticker', 'Bucket']).mean()['G_Spread_diff'].round(1).unstack(fill_value=np.nan,level=1).transpose()     
   


    elif aSpreadType == 'G Spread' and aCurncy in ['CZK','JPY']:
        if aType == 'Current':
            df = aBonds.groupby(['Ticker', 'Bucket']).mean()['G_Spread'].round(1).unstack(fill_value=np.nan,level=1).transpose()
        else:
            df = aBonds.groupby(['Ticker', 'Bucket']).mean()['G_Spread_diff'].round(1).unstack(fill_value=np.nan,level=1).transpose()     
                        
    elif aSpreadType == 'XCCY ASW':
        if aType == 'Current':
            if aXccyValue == 'Absolute' :
                df = aBonds.groupby(['Ticker', 'Bucket']).mean()['Xccy'].round(1).unstack(fill_value=np.nan,level=1).transpose()

            else :
                df = aBonds.groupby(['Ticker', 'Bucket']).mean()['Xccy'].round(1).unstack(fill_value=np.nan,level=1).transpose()
                dx = aBonds.groupby(['Ticker', 'Bucket']).mean()['i_spread'].round(1).unstack(fill_value=np.nan,level=1).transpose()
                df = (df - dx).round(1)
            try : 
                df ['TII'] = np.nan
            except :
                pass
            try : 
                df ['UKTI'] = np.nan
            except :
                pass
        else :
            df = aBonds.groupby(['Ticker', 'Bucket']).mean()['Xccy_Diff'].round(1).unstack(fill_value=np.nan,level=1).transpose()     
                              
            
    retreive_tickers = {'USD':['T','TII','IBRD','EIB','ESM','CADES','AGFRNC','KFW','RENTEN','NRW','NRWBK','BNG','KBN','KOMINS','KOMMUN','BMO','BNS','CM'],
          'EUR':['DBR','NETHER','FRTR','BGB','SPGB','BTPS','EIB','EU','ESM','EFSF','CADES','AGFRNC','UNEDIC','KFW','RENTEN','NRW','NRWBK','BNG','FLEMSH','KBN','KOMINS','KOMMUN','BMO','BNS','CM'],
          'GBP':['UKT','UKTI','KFW','ASIA','EIB','IBRD','DBR','CADES','RENTEN','NRW','NRWBK'],
          'JPY':['JGB','JFM','JICA'],
          'CZK':['CZGB','IBRD']
         }
    
    set_values = {
        'USD': {'Core Issuers' : ['T','TII','IBRD','EIB','KFW'], 'Sovereign':['T','TII'] ,'Supra & Agencies':['IBRD','EIB','ESM','CADES','AGFRNC','KFW','RENTEN','NRW','NRWBK','BNG','KBN','KOMINS','KOMMUN','BMO','BNS','CM']} , 
        'EUR': {'Core Issuers' : ['FRTR', 'DBR', 'EU', 'KFW'], 'Sovereign':['DBR','NETHER','FRTR','BGB','SPGB','BTPS'] ,
                'Supra & Agencies':['KFW','RENTEN','NRW','NRWBK','AGFRNC','CADES','UNEDIC','BNG','FLEMSH','EIB','EU','ESM','EFSF','KBN','KOMINS','KOMMUN','BMO','BNS','CM'] }
         
    }
    if len(list(atickers)) == 0 :
        orderTickers = retreive_tickers[aCurncy]
    elif 'ALL' not in list(atickers):
        orderTickers = list(atickers)
        if len(orderTickers)==1 and orderTickers[0][:4] not in ['Core','Supr','Sove'] :
            orderTickers = orderTickers + orderTickers
        elif orderTickers[0][:4] in ['Core','Supr','Sove']:
            orderTickers = set_values[aCurncy][orderTickers[0]]
    else :
        orderTickers = retreive_tickers[aCurncy]
    df = df.reindex(columns = orderTickers)
    df = df.reindex(bdBuckets)

    # df = df[bdTickers]
    # df = df.drop('1Y')
    return df

In [11]:
def Maximum_ASW(aCurrency,aRelative):
    df = runRequest(
        getBondsUniverse(aCurrency),
        {
        'Ticker': bq.data.ticker(),
        'I spread': bq.data.spread(pricing_source = 'BGN', side = 'Mid', spread_type='i',dates=bq.func.range(aRelative, '0D')).max(),
        'I spread Tod': bq.data.spread(pricing_source = 'BGN', side = 'Mid', spread_type='i'),
        'Bucket' : bq.func.bins(((bq.data.maturity() - bq.data.spread(pricing_source = 'BGN', side = 'Mid', spread_type='i',dates=bq.func.range(aRelative, '0D')).max()['DATE'])/365.25).round(0), bins=[1.5,2.5,3.5,4.5,5.5,6.5,7.5,8.5,9.5,10.5], bin_names=bdBuckets)
           }
    )
    if aCurrency == 'EUR':
        df = df [df['I spread Tod']>20]
        df['%'] = df['I spread Tod'] / df['I spread']
        df = df.groupby(['Ticker', 'Bucket']).mean()['%'].round(2).unstack(fill_value=np.nan,level=1).transpose()
    else :
        df = df.groupby(['Ticker', 'Bucket']).mean()['I spread'].round(2).unstack(fill_value=np.nan,level=1).transpose()
    return df

In [12]:
def plotHeatMap(aBonds):
    plt.figure(figsize = (6,6))
    plt.style.use("seaborn")
    sns.set(font_scale=1.6)
    heat_map = sns.heatmap(aBonds , annot = True,cmap ='PiYG',vmin=-50,vmax=50)
    # bottom, top = heat_map.get_ylim()
    # heat_map.set_ylim(bottom + 0.5, top - 0.5)
    
    return heat_map


global mBonds
global df
mBonds = pd.DataFrame()
global boxRow_5
global boxRow_6
global clickedX
clickedX = ''
global clickedY
clickedY = ''

def updateGlobalClicked(aX,aY):
    global clickedX
    clickedX = aX
    global clickedY
    clickedY = aY
    
    if len(mBonds.index) != 0:
        df_displayed = prepareDisplayDf(mBonds,clickedY,clickedX,'Difference')
        data_table = DataGrid(dataframe=df_displayed,
                        selection_mode = 'cell',
                        base_column_size=65,
                        column_widths ={'ISIN':110,
                                       'Description':150,
                                       'Issue_Date':80,
                                       'Maturity':80,
                                       'Amt_outstanding':100,
                                       'Axes_Ask':80,
                                       'Axes_Bid':80},
                             
                        layout={"height":"120px"})
        data = df.reset_index()
        chart_df = data[[df_displayed['Ticker'][0],'Bucket']]



        with fig.batch_update():
            fig.data[0].x = chart_df.iloc[:, 1]
            fig.data[0].y = chart_df.iloc[:, 0]
            fig.data[0].text = chart_df.iloc[:, 0]
            fig.data[0].name = f'Spread of '+ df_displayed['Ticker'][0]

    else:
        data_table = pd.DataFrame()
        boxRow_6.children = []
        boxRow_5.children = []
    boxRow_5.children = [data_table]
    boxRow_6.children = [fig]

    return 0

# Create some random data to work with
np.random.seed(0)
rows = ['Y Category {}'.format(i+1) for i in range(6)]
columns = ['X Category {}'.format(i+1) for i in range(15)]
dataframe = pd.DataFrame(np.random.randn(6,15), index=rows, columns=columns)

# Create the scales
scale_color = ColorScale(colors=['#FF1E3E', '#1a1a1a', '#30C030'],mid=float(0))
scale_x = OrdinalScale(allow_padding=False)
scale_y = OrdinalScale(allow_padding=False)

# Create the grid heat map
mark_grid_map = GridHeatMap(color=dataframe,
                            scales={'color': scale_color,
                                    'row':scale_y,
                                    'column':scale_x},
                            row=dataframe.index,
                            column=dataframe.columns)

mark_grid_map.on_element_click(lambda self, x: updateGlobalClicked(x.get('data').get('row'),x.get('data').get('column')))
# mark_grid_map.on_element_click(lambda self, x: print(x.get('row')))
# mark_grid_map.on_element_click(lambda self, x: print(type(x.get('data').get('row'))))

# Add the text labels
label_df = pd.melt(dataframe.reset_index(), id_vars=dataframe.index.name or 'index')
mark_label = Label(x=label_df[label_df.columns[1]],
                  y=label_df[label_df.columns[0]],
                  text=['{0:.1f}'.format(val)
                        for val in label_df[label_df.columns[2]]],
                  align='middle',
                  font_weight='normal',
                  default_size=12,
                  colors=['white'],
                  scales={'x': scale_x, 'y':scale_y})

# Create the axes
axis_y = Axis(scale=scale_y,orientation='vertical')
axis_x = Axis(scale=scale_x,
              tick_style={'text-anchor': 'start'},
              tick_rotate=25)

# Create the figure
visualization = Figure(marks=[mark_grid_map, mark_label],
                       axes=[axis_y, axis_x],
                       padding_y=0.0,
                       fig_margin={'top': 0, 'bottom':30,
                                   'left':30, 'right':10},
                       layout={'width': 'auto', 'height': '300px'},
                       title_style={'fill': 'white', 'font-size': '18'})


In [13]:
def update_plot(new_df,aType,Spread_Type):
    """This function will update the plot when provided a new dataframe."""
    if aType == 'Current':
        if Spread_Type == 'ASW Level Relative to' :
            scale_color = ColorScale(colors=['#FF1E3E', '#1a1a1a', '#4ec74e'],mid=float(60))
        else : 
            scale_color = ColorScale(colors=['#FF1E3E', '#1a1a1a', '#30C030'],mid=float(0))
    else:
        scale_color = ColorScale(colors=['#FF1E3E', '#1a1a1a', '#30C030'],mid=float(0))
        
    mark_grid_map.scales={'color': scale_color,'row':scale_y,'column':scale_x}
    mark_grid_map.color = new_df
    mark_grid_map.row = new_df.index
    mark_grid_map.column = new_df.columns
    new_label_df = pd.melt(new_df.reset_index(), id_vars=new_df.index.name or 'index')
    mark_label.x = new_label_df[new_label_df.columns[1]]
    mark_label.y = new_label_df[new_label_df.columns[0]]
    mark_label.text = ['{0:.1f}'.format(val)
                       for val in new_label_df[new_label_df.columns[2]]]


In [14]:
def prepareDisplayDf(df, ticker, bucket, display):
    
    data = df.copy()
    data = data[data['Ticker']==ticker]
    data = data[data['Bucket']==bucket]
    
    data = data.drop(['Date_C','Date_P','ResidualMaturity','ResidualMaturity_Past','Bucket','Bucket_Past',
                     'InterpolatedSwap','InterpolatedSwap_Past'],axis=1)
    
    data = data.round({'Yield_C':3,
                      'Yield_P':3
                      })
    
    
    data['ESG'] = data.apply(lambda row: defineESG(row), axis=1)
    
    data = data.drop(['Green','Social','Sustainable'],axis=1)
    # data = data.reset_index(drop=True)
    # data = data.set_index('ISIN')
    
    if display=='Current':
        data = data.drop(['Yield_P','Yield_Delta'],axis=1)
        
    data['Maturity'] = data['Maturity'].dt.strftime('%Y-%m-%d')
    data['Issue_Date'] = data['Issue_Date'].dt.strftime('%Y-%m-%d')
    data['Amt_outstanding'] = data['Amt_outstanding'].map('{:,.0f}'.format)
    data['Axes_Bid'] = data['Axes_Bid'].map('{:,.1f}'.format)
    data['Axes_Ask'] = data['Axes_Ask'].map('{:,.1f}'.format)

    return data

In [15]:
def defineESG(row):
    if row['Green']:
        return 'Green'
    if row['Social']:
        return 'Social'
    if row['Sustainable']:
        return 'Sustai'
    return '.'


# Create the initial Plotly figure
fig = go.FigureWidget()

# Customize the layout with a dark theme
fig = fig.update_layout(
    title='Spread by Tenor',
    xaxis_title='Tenor',
    yaxis_title='Spread',
    template='plotly_dark',
    font=dict(color='white'),
    xaxis=dict(
        showline=True,
        showgrid=False,
        linecolor='white',
        linewidth=1
    ),
    yaxis=dict(
        showline=True,
        showgrid=True,
        gridcolor='gray',
        linecolor='white'
    ),
    legend=dict(
        x=0.1, y=1.1,
        orientation="h",
        bgcolor='rgba(0,0,0,0)',
        bordercolor='white'
    ),
    width=800, height=500  
)

# Add the first empty trace which will be filled in update_chart()
fig = fig.add_trace(go.Scatter(
    mode='lines+markers+text', 
    line=dict(color='#a6c2e0', width=3),  
    marker=dict(color='#a6c2e0'),  
    textposition='top center', 
    textfont=dict(size=10)  
))

# Add the second empty trace which will be filled in update_chart()
fig = fig.add_trace(go.Scatter(
    mode='lines+markers+text', 
    line=dict(color='blue', width=3), 
    marker=dict(color='blue'),  
    textposition='bottom center',  
    textfont=dict(size=10)  
))


In [16]:
def update_chart(df,ticker,change=None):
    """
    Utility function for both initializing and updating the chart.
    """
    data = df.reset_index()
    if not change: # Initialize chart
        chart_df = data[['DBR','Bucket']]
    else: # Update chart
        chart_df = data[[ticker[0],'Bucket']]
        


    with fig.batch_update():
        fig.data[0].x = chart_df.iloc[:, 1]
        fig.data[0].y = chart_df.iloc[:, 0]
        fig.data[0].text = chart_df.iloc[:, 0]
        fig.data[0].name = f'Spread of {ticker[0]}'

In [17]:
curvesDictionary = {'ESTR':'YCSW0514'}
curvesDictionary['E3M'] = 'YCSW0201'
curvesDictionary['E6M'] = 'YCSW0045'
curvesDictionary['SOFR'] = 'YCSW0490'
curvesDictionary['FFUND'] = 'YCSW0042'
curvesDictionary['SONIA'] = 'YCSW0141'
curvesDictionary['JPY OIS'] = 'YCSW0195'
curvesDictionary['CZK RFR'] = 'YCSW0551'

parameters={'USD':['SOFR','FFUND'],'EUR':['ESTR','E3M','E6M'],'GBP':['SONIA'],'JPY':['JPY OIS'],'CZK':['CZK RFR']}

tickers ={'USD':['ALL','Core Issuers','Sovereign' ,'Supra & Agencies','T','TII','IBRD','EIB','ESM','CADES','AGFRNC','KFW','RENTEN','NRW','NRWBK','BNG','KBN','KOMINS','KOMMUN','BMO','BNS','CM'],
          'EUR':['ALL','Core Issuers','Sovereign' ,'Supra & Agencies','DBR','NETHER','FRTR','BTF','BGB','SPGB','BTPS','EIB','EU','ESM','EFSF','CADES','AGFRNC','UNEDIC','KFW','RENTEN','NRW','NRWBK','BNG','FLEMSH','KBN','KOMINS','KOMMUN','BMO','BNS','CM'],
          'GBP':['ALL','UKT','UKTI','KFW','ASIA','EIB','IBRD','DBR','CADES','RENTEN','NRW','NRWBK'],
          'JPY':['ALL','JGB','JFM','JICA'],
          'CZK':['ALL','CZGB','IBRD']
         }
Xccy = {'USD':['EUR ESTR','GBP SONIA'],'EUR':['USD SOFR','GBP SONIA'],'GBP':['USD SOFR','EUR ESTR'],'JPY':['USD SOFR','EUR ESTR'],'CZK':['USD SOFR','GBP SONIA','EUR ESTR']}
button_run = widgets.Button(description='Run')
wdg_ddwn_Ccy = widgets.Dropdown(options = parameters.keys(),description='Currency:')
wdg_ddwn_Index = widgets.Dropdown(description='Index:')
wdg_ddwn_Type = widgets.Dropdown(options = ['Current','Difference'],description='Analysis:')
wdg_ddwn_Histo = widgets.Dropdown(options = ['Manual','-1D','-2D','-3D','-1W','-2W','-3W','-1M','-3M','-6M','-9M','-1Y'],description='Compared to:')
wdg_ddwn_xccy_value = widgets.Dropdown(options = ['Absolute','Relative'],description='Value:')
wdg_date_picker = widgets.DatePicker(description='Date')
wdg_ticker = widgets.SelectMultiple(description='Tickers', disabled=False)
# data_table = DataGrid(dataframe=pd.DataFrame()) 
wdg_ddwn_Spread_Type = widgets.Dropdown(options = ['ASW','G Spread','XCCY ASW','ASW Level Relative to'],description='Spread Type:')   
wdg_relative = widgets.Dropdown(options = ['-1D','-2D','-3D','-1W','-2W','-3W','-1M','-3M','-6M','-9M','-1Y','-2Y'],description='Max:')   
wdg_XCCY = widgets.Dropdown(description='XCCY ASW', disabled=False)
status_label = widgets.Label()
status_label.layout.width = '400px'

dates_label = widgets.Label()
dates_label.layout.width = '400px'

boxRow_1 = widgets.HBox([wdg_ddwn_Ccy, wdg_ddwn_Index,wdg_ticker])
boxRow_2 = widgets.HBox([wdg_ddwn_Type, wdg_ddwn_Histo,wdg_ddwn_Spread_Type,wdg_XCCY,wdg_relative])
boxRow_3 = widgets.HBox()
boxRow_4 = widgets.HBox([button_run, dates_label])
boxRow_5 = widgets.HBox([])
boxRow_6 = widgets.HBox([])

main_box = widgets.VBox([boxRow_1, boxRow_2, boxRow_3, boxRow_4, status_label, visualization,boxRow_5,boxRow_6])


def update_wdgIndex_options(*args):
    wdg_ddwn_Index.options = parameters[wdg_ddwn_Ccy.value]
    wdg_ticker.options = tickers[wdg_ddwn_Ccy.value]
    
    
def update_wdgType_options(*args): 
    wdg_XCCY.options = Xccy[wdg_ddwn_Ccy.value]
    if wdg_ddwn_Type.value == 'Current':
        if wdg_ddwn_Spread_Type.value != 'XCCY ASW':
            if wdg_ddwn_Spread_Type.value == 'ASW Level Relative to' :
                boxRow_2.children = [wdg_ddwn_Type,wdg_ddwn_Spread_Type,wdg_relative]
            else :
                boxRow_2.children = [wdg_ddwn_Type,wdg_ddwn_Spread_Type]
        else :
            boxRow_2.children = [wdg_ddwn_Type,wdg_ddwn_Spread_Type,wdg_XCCY,wdg_ddwn_xccy_value]
    else:
        if wdg_ddwn_Spread_Type.value != 'XCCY ASW':
            boxRow_2.children = [wdg_ddwn_Type,wdg_ddwn_Histo,wdg_ddwn_Spread_Type]
        else : 
            boxRow_2.children = [wdg_ddwn_Type,wdg_ddwn_Histo,wdg_ddwn_Spread_Type,wdg_XCCY]
    
def update_wdgHisto_options(*args):
    if wdg_ddwn_Histo.value == 'Manual':
        boxRow_3.children = [wdg_date_picker]
    else:
        boxRow_3.children = []



wdg_ddwn_Ccy.observe(update_wdgIndex_options)
wdg_ddwn_Type.observe(update_wdgType_options)
wdg_ddwn_Histo.observe(update_wdgHisto_options)
wdg_ddwn_Spread_Type.observe(update_wdgType_options)
wdg_ddwn_Ccy.observe(update_wdgType_options)

wdg_ddwn_Ccy.value='EUR'
wdg_ddwn_Type.value='Difference'
wdg_ddwn_Histo.value='-1D'
wdg_ticker.value= ('Core Issuers',)

def button_run_on_click(_):
    
    status_label.value = 'Running...'
    
    if wdg_ddwn_Histo.value == 'Manual':
        mDate = wdg_date_picker.value
    else:
        mDate = wdg_ddwn_Histo.value
        # If not business day, moving to the nearest previous business day
        hsbc = 'EESWE1 Curncy'
        price = bq.data.px_last(dates = bq.func.range(mDate,'0d'))['DATE']
        minday = price.min()
        req = bql.Request(hsbc, {'date':minday})
        res = bq.execute(req)
        data = res[0].df()
        mDate = data.iloc[0]['date'].date()
        if not(mDate == ((mDate - BDay(0)).date())):
            mDate = (mDate - BDay(1)).date()
    global mBonds
    global df
    mBonds = getBonds(mDate,wdg_ddwn_Ccy.value,wdg_XCCY.value)
    if wdg_ddwn_Spread_Type.value != 'XCCY ASW' :
        mSwaps = getSwaps(curvesDictionary[wdg_ddwn_Index.value],mDate)
    else :
        mSwaps = getSwaps(curvesDictionary[parameters[wdg_XCCY.value[0:3]][0]],mDate)
        
    mBonds = computeISpread(mBonds,mSwaps,wdg_ddwn_Spread_Type.value)
    df = aggregateBonds(mBonds,wdg_ddwn_Type.value,wdg_ddwn_Ccy.value,wdg_ticker.value,wdg_ddwn_Spread_Type.value,wdg_relative.value,wdg_ddwn_xccy_value.value)
    update_plot(df,wdg_ddwn_Type.value,wdg_ddwn_Spread_Type.value)
    #Updating
    
    
    status_label.value = ''
    dates_label.value = 'From ' + mBonds.iloc[0]['Date_P'].strftime("%d-%b-%Y") + ' To ' + mBonds.iloc[0]['Date_C'].strftime("%d-%b-%Y") + ' . As of ' + datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    
    
button_run.on_click(button_run_on_click)

display(main_box)